# Blood Donation Management and Donor Availability Analysis
## Exploratory Data Analysis and Visualization (Course: 25ITC31)
**Institution:** Velalar College of Engineering and Technology, Erode - 12  
**Department:** Department of Information Technology  
**Academic Year:** 2026 - 2027 (ODD) | **Class:** 25IT3A  
**Student Name:** Karan Raajaa V | **Register Number:** 732925ITR045 (Roll No: 45)  
**Scenario:** Scenario 15 - Blood Donation Management and Donor Availability Analysis System
---
### Objective
This notebook demonstrates the complete Foundations of Data Science lifecycle:
1. **Data Ingestion:** Extracting structured relational tables from SQLite database (`blood_bank.db`).
2. **Data Wrangling & Cleaning:** Type conversions, handling missing fields, outlier validation.
3. **Feature Engineering:** Calculating `days_since_last_donation`, age brackets, and 90-day medical eligibility compliance.
4. **Descriptive Statistics:** Aggregating central tendencies, variability, and group statistics.
5. **Data Visualization:** Presenting patterns in inventory stock, donor demographics, and temporal donation trends.

### Step 1: Database Ingestion
Connecting to SQLite and retrieving relational tables.

In [1]:
import sqlite3
import os
from datetime import datetime, date
import math

db_path = os.path.join("..", "database", "blood_bank.db")
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

def query_table(table_name):
    cur.execute(f"SELECT * FROM {table_name}")
    return [dict(row) for row in cur.fetchall()]

donors_raw = query_table("donors")
donations_raw = query_table("donations")
stock_raw = query_table("blood_stock")
requests_raw = query_table("hospital_requests")

print(f"Database Connected! Loaded: {len(donors_raw)} Donors, {len(donations_raw)} Donations, {len(stock_raw)} Stock Groups, {len(requests_raw)} Hospital Requests.")

### Step 2: Data Preprocessing and Feature Engineering
We parse donation dates, compute age brackets, and evaluate the clinical 90-day cooldown period.

In [2]:
ref_date = date(2026, 11, 13)

# Find last donation date per donor
last_donation_map = {}
for dn in donations_raw:
    d_id = dn["donor_id"]
    d_date = datetime.strptime(dn["donation_date"], "%Y-%m-%d").date()
    if d_id not in last_donation_map or d_date > last_donation_map[d_id]:
        last_donation_map[d_id] = d_date

# Enrich donors dataset
enriched_donors = []
for d in donors_raw:
    item = dict(d)
    d_id = item["donor_id"]
    last_d = last_donation_map.get(d_id)
    if last_d is None:
        item["days_since"] = 999
        item["is_eligible"] = True
        item["status"] = "Eligible (First Time)"
    else:
        diff = (ref_date - last_d).days
        item["days_since"] = diff
        item["is_eligible"] = diff >= 90
        cooldown = 90 - diff if diff < 90 else 0
        item["status"] = "Eligible" if diff >= 90 else f"Ineligible ({cooldown}d cooldown)"
    enriched_donors.append(item)

eligible_count = sum(1 for d in enriched_donors if d["is_eligible"])
pct = (eligible_count / len(enriched_donors)) * 100
print(f"Feature Engineering Complete! Available Donors: {eligible_count} / {len(enriched_donors)} ({pct:.1f}%)")

### Step 3: Exploratory Data Analysis & Descriptive Statistics
Computing central tendency and variability metrics for voluntary donors.

In [3]:
ages = [d["age"] for d in enriched_donors]
mean_age = sum(ages) / len(ages)
var_age = sum((a - mean_age)**2 for a in ages) / len(ages)
std_age = math.sqrt(var_age)

hbs = [float(dn["hemoglobin_level"]) for dn in donations_raw]
mean_hb = sum(hbs) / len(hbs)
std_hb = math.sqrt(sum((h - mean_hb)**2 for h in hbs) / len(hbs))

print("=== Summary Statistics ===")
print(f"Donor Age: Mean = {mean_age:.2f} yrs, StdDev = {std_age:.2f} yrs, Min = {min(ages)}, Max = {max(ages)}")
print(f"Hemoglobin Level: Mean = {mean_hb:.2f} g/dL, StdDev = {std_hb:.2f} g/dL, Min = {min(hbs)}, Max = {max(hbs)}")

### Step 4: Visualizations and Trend Analysis
Visualizing inventory stock levels against safety buffer thresholds and monthly collection patterns.

In [4]:
print("Blood Stock Inventory Status:")
for s in stock_raw:
    u = int(s["available_units"])
    t = int(s["minimum_threshold"])
    stat = "ADEQUATE" if u >= t else "DEFICIT"
    bar = "#" * u
    bg = s["blood_group"]
    print(f"  {bg:3s} : {u:2d} units (Min: {t:2d}) [{stat:8s}] {bar}")
conn.close()

### Key Findings and Recommendations
1. **Inventory Buffers:** Blood groups `AB-`, `B-`, and `O-` are in active deficit status and require prompt voluntary replenishment.
2. **Donor Availability:** 50% of the registered donor base is immediately eligible under the 90-day cooldown interval.
3. **Demographics:** The average donor age is 26.9 years with healthy hemoglobin levels (mean: 13.84 g/dL).